# EMC Logbook — Time-Series Dataset Cleaning
### Merges W_out_Act + W_Act, deduplicates, filters excluded venues/activities, and produces a clean monthly reservation count series for Holt-Winters forecasting.


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")


## 1. Load raw logbook files

In [ ]:
df_no_act = pd.read_csv('EMC_Logbook_-_W_out_Act.csv')
df_act    = pd.read_csv('EMC_Logbook_-_W_Act.csv')

print(f"W_out_Act shape : {df_no_act.shape}")
print(f"W_Act shape     : {df_act.shape}")


## 2. Normalise column names

In [ ]:
def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

df_no_act = clean_columns(df_no_act)
df_act    = clean_columns(df_act)

print("W_out_Act columns:", df_no_act.columns.tolist())
print("W_Act columns    :", df_act.columns.tolist())


## 3. Align schemas and merge

In [ ]:
# W_out_Act has no ACTIVITY column — add it as NaN so schemas match
if 'activity' not in df_no_act.columns:
    df_no_act['activity'] = np.nan

# Keep only the shared columns we need
KEEP_COLS = ['request_date', 'requestor_name', 'department', 'venue', 'activity', 'contact_number']

df_no_act = df_no_act[[c for c in KEEP_COLS if c in df_no_act.columns]]
df_act    = df_act[[c for c in KEEP_COLS if c in df_act.columns]]

# Tag source so we can inspect provenance if needed
df_no_act['_source'] = 'logbook_no_act'
df_act['_source']    = 'logbook_act'

df_raw = pd.concat([df_no_act, df_act], ignore_index=True)
print(f"Combined rows before cleaning: {len(df_raw)}")
df_raw.head(3)


## 4. Parse and validate dates

In [ ]:
def parse_date(val):
    """Try multiple format patterns; return NaT if nothing works."""
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip().rstrip('/')   # fix trailing slash e.g. '09/03/25/'
    for fmt in ('%m/%d/%y', '%m/%d/%Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except ValueError:
            pass
    return pd.NaT

df_raw['date'] = df_raw['request_date'].apply(parse_date)

invalid_dates = df_raw['date'].isna().sum()
print(f"Rows with unparseable dates : {invalid_dates}")
print(f"Date range (raw)            : {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")

# Drop rows with no parseable date — they cannot contribute to the time-series
df_raw = df_raw.dropna(subset=['date']).copy()
print(f"Rows after dropping bad dates: {len(df_raw)}")


## 5. Remove obviously wrong years (typos)

In [ ]:
# Keep only dates within a plausible operating window (2024-01-01 → today)
VALID_START = pd.Timestamp('2024-01-01')
VALID_END   = pd.Timestamp.today().normalize()

before = len(df_raw)
df_raw = df_raw[(df_raw['date'] >= VALID_START) & (df_raw['date'] <= VALID_END)].copy()
print(f"Removed {before - len(df_raw)} rows outside {VALID_START.date()} – {VALID_END.date()}")
print(f"Remaining rows: {len(df_raw)}")


## 6. Normalise text fields

In [ ]:
def norm_str(series):
    """Strip whitespace and upper-case for consistent comparisons."""
    return series.fillna('').astype(str).str.strip().str.upper()

df_raw['requestor_name_n'] = norm_str(df_raw['requestor_name'])
df_raw['department_n']     = norm_str(df_raw['department'])
df_raw['venue_n']          = norm_str(df_raw['venue'])
df_raw['activity_n']       = norm_str(df_raw['activity'])
df_raw['contact_n']        = norm_str(df_raw['contact_number'])

# Collapse venue spelling variants (QUANDRANGLE → QUADRANGLE, etc.)
venue_map = {
    'QUANDRANGLE' : 'QUADRANGLE',
    'OTHERS(EMC)' : 'OTHERS',
    'OTHERS '     : 'OTHERS',
    'OTHER'       : 'OTHERS',
}
df_raw['venue_n'] = df_raw['venue_n'].replace(venue_map)

print("Venue values after normalisation:", sorted(df_raw['venue_n'].unique()))


## 7. Filter out excluded venue / activity keywords

Remove any row whose **VENUE** or **ACTIVITY** field contains references to:
`gym`, `sports`, `pool`, `med auditorium`, `BED mini auditorium`, `OVAL`, `mini auditorium`


In [ ]:
EXCLUDE_PATTERNS = [
    r'\bgym\b',
    r'\bsports\b',
    r'\bpool\b',
    r'\boval\b',
    r'med[\s_-]*audi',          # med auditorium / med audi
    r'medical[\s_-]*audi',
    r'bed[\s_-]*mini',          # BED mini auditorium
    r'mini[\s_-]*audi',         # mini auditorium (any prefix)
    r'mini[\s_-]*gym',          # mini gym
]

combined_pattern = '|'.join(EXCLUDE_PATTERNS)

venue_hit    = df_raw['venue_n'].str.contains(combined_pattern, flags=re.IGNORECASE, regex=True)
activity_hit = df_raw['activity_n'].str.contains(combined_pattern, flags=re.IGNORECASE, regex=True)
exclude_mask = venue_hit | activity_hit

excluded_rows = df_raw[exclude_mask].copy()
df_clean      = df_raw[~exclude_mask].copy()

print(f"Rows excluded by keyword filter : {exclude_mask.sum()}")
print(f"Rows remaining                  : {len(df_clean)}")
print()
print("Sample excluded activities:")
print(excluded_rows['activity_n'].value_counts().head(15).to_string())


## 8. Remove duplicates

A record is a duplicate if it shares the same **date + requestor name + department + venue**.
When W_Act provides an ACTIVITY for a row already present in W_out_Act, we keep the
richer W_Act version (higher _source priority).


In [ ]:
# Sort so that W_Act rows (with ACTIVITY) are preferred over W_out_Act rows
source_order = {'logbook_act': 0, 'logbook_no_act': 1}
df_clean['_sort_key'] = df_clean['_source'].map(source_order)
df_clean = df_clean.sort_values('_sort_key')

DEDUP_KEYS = ['date', 'requestor_name_n', 'department_n', 'venue_n']

before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=DEDUP_KEYS, keep='first').copy()

print(f"Rows before dedup : {before_dedup}")
print(f"Rows after dedup  : {len(df_clean)}")
print(f"Duplicates removed: {before_dedup - len(df_clean)}")


## 9. Secondary dedup — same date + contact number

Contact numbers sometimes differ slightly between the two logbooks
(e.g. `63+9234651282` vs `63+9234651282` with extra spaces).
We normalise them and do a second pass.


In [ ]:
# Normalise contact: keep digits only
df_clean['contact_digits'] = df_clean['contact_n'].str.replace(r'\D', '', regex=True)

DEDUP_KEYS_2 = ['date', 'contact_digits', 'venue_n']
before2 = len(df_clean)

# Only deduplicate where contact_digits is non-empty (avoid collapsing blanks)
has_contact = df_clean['contact_digits'].str.len() >= 7
df_with    = df_clean[has_contact].drop_duplicates(subset=DEDUP_KEYS_2, keep='first')
df_without = df_clean[~has_contact]

df_clean = pd.concat([df_with, df_without], ignore_index=True)
print(f"Rows before secondary dedup : {before2}")
print(f"Rows after secondary dedup  : {len(df_clean)}")
print(f"Additional duplicates removed: {before2 - len(df_clean)}")


## 10. Preview cleaned dataset

In [ ]:
display_cols = ['date', 'requestor_name', 'department', 'venue', 'activity', '_source']
df_clean[display_cols].sort_values('date').head(10)


## 11. Aggregate to monthly reservation counts

In [ ]:
df_clean['month'] = df_clean['date'].dt.to_period('M')

monthly = (
    df_clean
    .groupby('month')
    .size()
    .reset_index(name='reservation_count')
)
monthly['month'] = monthly['month'].astype(str)

# Fill any missing months in the range with 0 so the series is contiguous
all_months = pd.period_range(
    start=df_clean['date'].min().to_period('M'),
    end=df_clean['date'].max().to_period('M'),
    freq='M'
)
full_index = pd.DataFrame({'month': all_months.astype(str)})
monthly = full_index.merge(monthly, on='month', how='left').fillna(0)
monthly['reservation_count'] = monthly['reservation_count'].astype(int)

print(f"Monthly series length: {len(monthly)} months")
print()
print(monthly.to_string(index=False))


## 12. Save outputs

In [ ]:
# Full cleaned record-level dataset
df_export = df_clean[['date', 'requestor_name', 'department', 'venue', 'activity', '_source']].copy()
df_export.to_csv('EMC_Cleaned_Records.csv', index=False)

# Final monthly time-series (the input for Holt-Winters)
monthly.to_csv('EMC_Monthly_Reservations_Cleaned.csv', index=False)

print("Saved:")
print("  EMC_Cleaned_Records.csv            — full cleaned record-level dataset")
print("  EMC_Monthly_Reservations_Cleaned.csv — monthly time-series for Holt-Winters")


## 13. Quick plot — monthly reservation trend

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly['month'], monthly['reservation_count'], color='steelblue', alpha=0.85)
ax.set_xlabel('Month')
ax.set_ylabel('Reservations')
ax.set_title('EMC Monthly Reservations — Cleaned Dataset')
ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('EMC_Monthly_Trend.png', dpi=150)
plt.show()
print("Plot saved as EMC_Monthly_Trend.png")


## 14. Summary statistics

In [ ]:
print("=== Cleaning summary ===")
print(f"Total rows loaded          : {len(df_no_act) + len(df_act)}")
print(f"Rows after date validation : see step 4")
print(f"Rows after year filter     : see step 5")
print(f"Rows after keyword filter  : see step 7")
print(f"Final cleaned rows         : {len(df_clean)}")
print()
print("=== Time-series summary ===")
print(f"Months in series           : {len(monthly)}")
print(f"Date range                 : {monthly['month'].iloc[0]} → {monthly['month'].iloc[-1]}")
print(f"Mean reservations/month    : {monthly['reservation_count'].mean():.1f}")
print(f"Max reservations/month     : {monthly['reservation_count'].max()} ({monthly.loc[monthly['reservation_count'].idxmax(), 'month']})")
print(f"Min reservations/month     : {monthly['reservation_count'].min()} ({monthly.loc[monthly['reservation_count'].idxmin(), 'month']})")
print()
print("=== Tip for Holt-Winters ===")
print("Feed 'EMC_Monthly_Reservations_Cleaned.csv' into statsmodels ExponentialSmoothing")
print("Use trend='add', seasonal='add', seasonal_periods=12 once you have 24+ months.")
